# CFSPMNet full LOSO evaluation on Liu2024

Runs the two-stage SPPM training loop (`02_cfspmnet_sppm_training.ipynb`)
across a full leave-one-subject-out sweep and reports paper-Table-3-style
aggregate metrics (accuracy, kappa, F1, precision, recall, ROC AUC, mean +/- std
across folds).

**Runtime warning**: this is 50 folds of two-stage training (Stage I
pretraining + Stage II joint adaptation), each on ~49 source subjects' worth
of windows. That is considerably more compute than the 3-subject smoke test
in notebook 2. **Always run with `SMOKE=True` first** to confirm the loop,
checkpointing, and CSV output all work, before switching to `SMOKE=False` and
letting the full sweep run (likely as a long background job rather than
interactively cell-by-cell -- see the runtime note in Part 1).

Cohort note: this uses the **full 50-subject** Liu2024 cohort wired up in
`configs/dataset/liu2024.yaml`, not the paper's clinically-screened 24-subject
XW-Stroke subcohort (right-handed, first-ever stroke, NIHSS<10, duration<10y).
The aggregate numbers here are therefore a project-native LOSO result, not a
like-for-like reproduction of the paper's reported 68.23% +/- 5.13%.

In [1]:
import csv
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root")


repo_root = find_repo_root(Path.cwd().resolve())
repo_root

PosixPath('/workspaces/BRAINDECODE')

## 1. Configuration: `SMOKE` vs full sweep

`SMOKE=True` restricts the sweep to 3 subjects (3 folds, 2 source + 1 target
each) and a handful of epochs -- enough to validate the loop mechanics,
checkpoint paths, and CSV schema in well under a minute. `SMOKE=False` runs
all 50 subjects (50 folds) with the paper's XW-Stroke hyperparameters from
Table 2 (`alpha=0.98`, `pseudo_threshold=0.60`,
`matching_tolerance_floor=0.50`, `Stage-I epochs=25`). The paper does not
state an explicit Stage-II epoch count (only a 200-epoch overall cap, section
3.2) -- `STAGE_II_EPOCHS=50` below is this notebook's own choice, not a paper
value, and is a reasonable place to start tuning against a validation curve
like the one plotted in notebook 2.

In [2]:
SMOKE = True  # set False to run the full 50-subject sweep

if SMOKE:
    SUBJECT_IDS = [1, 2, 3]
    STAGE_I_EPOCHS = 3
    STAGE_II_EPOCHS = 3
    MATCHING_TOLERANCE_FLOOR = 0.30  # relaxed: see 02_cfspmnet_sppm_training.ipynb Part 3
else:
    SUBJECT_IDS = list(range(1, 51))
    STAGE_I_EPOCHS = 25  # paper Table 2, XW-Stroke column
    STAGE_II_EPOCHS = 50  # not specified by the paper; see note above
    MATCHING_TOLERANCE_FLOOR = 0.50  # paper Table 2 (DATASET_PRESETS["XW_30Chs"] in temp/sppm_strategy.py)

ALPHA = 0.98  # paper Table 2, XW-Stroke column
PSEUDO_THRESHOLD = 0.60  # paper Table 2
BATCH_SIZE = 40  # paper Table 2 ("XW_30Chs" preset)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
SEED = 2  # paper Table 2 ("XW_30Chs" preset)

print(f"SMOKE={SMOKE}, {len(SUBJECT_IDS)} subjects -> {len(SUBJECT_IDS)} LOSO folds")

SMOKE=True, 3 subjects -> 3 LOSO folds


## 2. Load all subjects once and materialize canonical trials

Loading + canonicalizing every subject's windows is done a single time,
outside the fold loop -- each fold below is then just numpy masking over
these dense in-memory arrays.

In [3]:
from eeg_bci.cfspmnet.canonicalization import load_liu2024_participants
from eeg_bci.cfspmnet.datasets import (
    build_loso_fold,
    chronological_adapt_test_indices,
    materialize_canonical_trials,
)
from eeg_bci.data.moabb import build_moabb_dataset
from eeg_bci.utils.seed import seed_everything

participants = load_liu2024_participants(
    repo_root / "data/moabb/MNE-liu2024-data/files/participants.tsv"
)

dataset_cfg = OmegaConf.load(repo_root / "configs/dataset/liu2024.yaml")
preprocessing_cfg = OmegaConf.load(repo_root / "configs/preprocessing/liu2024.yaml")
dataset_cfg.subject_ids = SUBJECT_IDS

t_load_start = time.time()
windows, dataset_info = build_moabb_dataset(dataset_cfg, preprocessing_cfg)
metadata = windows.get_metadata().reset_index(drop=True)
canonical_x, canonical_y, subject_ids = materialize_canonical_trials(windows, metadata, participants)
adapt_idx_all, test_idx_all = chronological_adapt_test_indices(windows, test_size=0.2, stratify=True)
print(f"PASS: loaded + canonicalized {len(canonical_x)} windows across {len(SUBJECT_IDS)} subjects in {time.time() - t_load_start:.1f}s")
dataset_info

Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


Used Annotations descriptions: [np.str_('left_hand'), np.str_('right_hand')]


/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:521: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


PASS: loaded + canonicalized 120 windows across 3 subjects in 8.0s


DatasetInfo(n_chans=29, n_outputs=2, n_times=2000, sfreq=500.0)

## 3. Output layout

Follows this project's `{dataset}/{method}/...` convention (see
`eeg_bci.tracking.results`): per-fold checkpoints under
`held_out_subject_<id>/model.pt` (matching `train_loso.py`'s naming), and a
single `loso_results.csv` with project-standard column ordering
(`eeg_bci.tracking.results.order_fieldnames`).

In [4]:
from eeg_bci.tracking.results import order_fieldnames

output_dir = repo_root / "outputs/results/liu2024/leave_one_subject_out/cfspmnet_sppm"
checkpoint_dir = output_dir / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_csv_path = output_dir / "loso_results.csv"
output_dir

PosixPath('/workspaces/BRAINDECODE/outputs/results/liu2024/leave_one_subject_out/cfspmnet_sppm')

## 4. LOSO loop: one fold per held-out subject

Per fold: build the source / target-adapt / target-test split, run Stage I
+ Stage II, evaluate on the never-adapted-on target-test partition, then
persist a checkpoint and a results row.

In [5]:
from eeg_bci.cfspmnet.model import CFSPMNet
from eeg_bci.cfspmnet.sppm import build_shared_private_signature_prototypes
from eeg_bci.cfspmnet.training import (
    SPPMTrainingConfig,
    build_sppm_dataloaders,
    evaluate_target,
    initialize_target_pseudo_labels,
    train_source_only_epoch,
    train_sppm_epoch,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(SEED)

rows = []
t_sweep_start = time.time()

for held_out_subject in SUBJECT_IDS:
    t_fold_start = time.time()

    fold = build_loso_fold(
        canonical_x, canonical_y, subject_ids, adapt_idx_all, test_idx_all, held_out_subject=held_out_subject
    )

    model = CFSPMNet.from_dataset_info(dataset_info).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    source_loader, target_adapt_loader, target_test_loader, target_init_loader = build_sppm_dataloaders(
        fold.source_dataset, fold.target_adapt_dataset, fold.target_test_dataset, batch_size=BATCH_SIZE
    )

    for _ in range(STAGE_I_EPOCHS):
        train_source_only_epoch(source_loader, model, optimizer, device)

    shared_prototypes, class_wise_matching_tolerance = build_shared_private_signature_prototypes(
        fold.source_signature_vectors, fold.source_labels, num_classes=2, floor=MATCHING_TOLERANCE_FLOOR
    )
    shared_prototypes_t = torch.as_tensor(shared_prototypes, dtype=torch.float32, device=device)
    tolerance_t = torch.as_tensor(class_wise_matching_tolerance, dtype=torch.float32, device=device)

    cfg = SPPMTrainingConfig(device=device, alpha=ALPHA, pseudo_threshold=PSEUDO_THRESHOLD)
    initialize_target_pseudo_labels(cfg, model, target_init_loader, shared_prototypes_t, tolerance_t)

    last_stage_ii_stats = None
    for _ in range(STAGE_II_EPOCHS):
        last_stage_ii_stats = train_sppm_epoch(
            cfg, source_loader, target_adapt_loader, model, optimizer, shared_prototypes_t, tolerance_t
        )

    result = evaluate_target(model, target_test_loader, device, dataset_info.n_outputs)

    fold_checkpoint_dir = checkpoint_dir / f"held_out_subject_{held_out_subject}"
    fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), fold_checkpoint_dir / "model.pt")

    row = {
        "held_out_subject": held_out_subject,
        "n_train_windows": len(fold.source_dataset),
        "n_target_adapt_windows": len(fold.target_adapt_dataset),
        "n_test_windows": len(fold.target_test_dataset),
        "final_accepted_ratio": last_stage_ii_stats["accepted_ratio"] if last_stage_ii_stats else 0.0,
        "final_active_ratio": last_stage_ii_stats["active_ratio"] if last_stage_ii_stats else 0.0,
        "fold_seconds": time.time() - t_fold_start,
        **{f"test_{name}": value for name, value in result["metrics"].items() if name != "confusion_matrix"},
    }
    rows.append(row)
    print(
        f"[{held_out_subject:>3}] fold {len(rows):>2}/{len(SUBJECT_IDS)} "
        f"test_accuracy={row['test_accuracy']:.3f} test_cohen_kappa={row['test_cohen_kappa']:.3f} "
        f"({row['fold_seconds']:.1f}s)"
    )

print(f"PASS: completed {len(rows)}/{len(SUBJECT_IDS)} LOSO folds in {time.time() - t_sweep_start:.1f}s")

/workspaces/BRAINDECODE/.venv/lib/python3.11/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1090.)
  return F.conv2d(


[  1] fold  1/3 test_accuracy=0.500 test_cohen_kappa=0.000 (6.2s)
[  2] fold  2/3 test_accuracy=0.250 test_cohen_kappa=-0.500 (4.4s)
[  3] fold  3/3 test_accuracy=0.500 test_cohen_kappa=0.000 (4.4s)
PASS: completed 3/3 LOSO folds in 14.9s


In [6]:
fieldnames = order_fieldnames(rows, leading=["held_out_subject"])
with results_csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"PASS: wrote {len(rows)} rows to {results_csv_path}")
results_df = pd.DataFrame(rows)
results_df.head()

PASS: wrote 3 rows to /workspaces/BRAINDECODE/outputs/results/liu2024/leave_one_subject_out/cfspmnet_sppm/loso_results.csv


,held_out_subject,n_train_windows,n_target_adapt_windows,n_test_windows,final_accepted_ratio,final_active_ratio,fold_seconds,test_accuracy,test_balanced_accuracy,test_cohen_kappa,test_macro_f1,test_macro_precision,test_macro_recall,test_roc_auc
0,1,80,32,8,0.265625,0.34375,6.182539,0.50,0.50,0.0,0.333333,0.250000,0.50,0.3125
1,2,80,32,8,0.328125,0.53125,4.390344,0.25,0.25,-0.5,0.200000,0.166667,0.25,0.1875
2,3,80,32,8,0.046875,0.15625,4.354675,0.50,0.50,0.0,0.333333,0.250000,0.50,0.7500


## 5. Aggregate paper-style summary

Mean +/- std across all folds, in the same shape as CFSPMNet paper Table 3/4.

In [8]:
SUMMARY_METRICS = [
    "test_accuracy",
    "test_cohen_kappa",
    "test_macro_f1",
    "test_macro_precision",
    "test_macro_recall",
    "test_roc_auc",
]

summary = pd.DataFrame(
    {
        "mean": results_df[SUMMARY_METRICS].mean(),
        "std": results_df[SUMMARY_METRICS].std(),
    }
)
print(f"CFSPMNet + SPPM, Liu2024 LOSO ({len(rows)} subjects, SMOKE={SMOKE})")
summary

CFSPMNet + SPPM, Liu2024 LOSO (3 subjects, SMOKE=True)


,mean,std
test_accuracy,0.416667,0.144338
test_cohen_kappa,-0.166667,0.288675
test_macro_f1,0.288889,0.076980
test_macro_precision,0.222222,0.048113
test_macro_recall,0.416667,0.144338
test_roc_auc,0.416667,0.295363


## Conclusion

The LOSO loop, per-fold checkpointing, and results CSV all run end to end
(validated here with `SMOKE=True`). Before trusting the aggregate numbers as
a real result: switch to `SMOKE=False` and let the full 50-subject sweep run
to completion (expect a long run -- consider `jupyter nbconvert --execute`
as a background job rather than running interactively), then compare against
`temp/CFSPMNET.md` Table 3's 68.23% +/- 5.13% keeping the cohort mismatch (50
vs. the paper's clinically-screened 24 subjects) in mind. A follow-up pass
could add the `VARIANT_CONFIGS` ablations from `temp/sppm_strategy.py`
(SourceOnly, NoSPPM, NoFRSMamba, ...) to reproduce the paper's Table 5.